# 02 - Descripción de los datos y limpieza


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = PROJECT_DIR / "data" / "raw"
PROC = PROJECT_DIR / "data" / "processed"

raw        = pd.read_csv(RAW / "train_annotations.csv")
events     = pd.read_csv(PROC / "events.csv")
plays      = pd.read_csv(PROC / "plays.csv")
video_meta = pd.read_csv(PROC / "video_meta.csv")
clip_index = pd.read_csv(PROC / "clip_index.csv")

for name, df in [("raw", raw), ("events", events), ("plays", plays),
                 ("video_meta", video_meta), ("clip_index", clip_index)]:
    print(f"{name:12s} {df.shape[0]:6d} filas x {df.shape[1]:2d} columnas")

raw           11218 filas x  4 columnas
events        11218 filas x 11 columnas
plays          3418 filas x  5 columnas
video_meta       12 filas x  8 columnas
clip_index      425 filas x  5 columnas


## 1. Origen y trazabilidad

La competencia de Kaggle está **cerrada**; los datos originales.
Se trabaja con dos *mirrors*:

| Qué | Fuente | Peso | En el repo |
|---|---|---|---|
| **Anotaciones** (`train_annotations.csv`) | HuggingFace `dbal0503/Bundesliga` (`train.csv`) | 460 KB | **Sí** |
| **Videos** | Kaggle `saberghaderi/-dfl-bundesliga-460-mp4-videos-in-30sec-csv` | ~10 GB | No |

`train_annotations.csv` es la **única fuente primaria**. Todo lo demás (`events`, `plays`,
`video_meta`, `clip_index`) se **deriva** de ese archivo (y del listado de clips del mirror
de video) en el notebook `01`.

In [2]:
# --- Anotaciones crudas: forma y tipos ---
print("shape:", raw.shape)
print("\nColumnas y tipo declarado:")
print(raw.dtypes)
print("\nPrimeras filas:")
raw.head(8)

shape: (11218, 4)

Columnas y tipo declarado:
video_id             object
time                float64
event                object
event_attributes     object
dtype: object

Primeras filas:


,video_id,time,event,event_attributes
0,1606b0e6_0,200.265822,start,NaN
1,1606b0e6_0,201.150000,challenge,['ball_action_forced']
2,1606b0e6_0,202.765822,end,NaN
3,1606b0e6_0,210.124111,start,NaN
4,1606b0e6_0,210.870000,challenge,['opponent_dispossessed']
5,1606b0e6_0,212.624111,end,NaN
6,1606b0e6_0,217.850213,start,NaN
7,1606b0e6_0,219.230000,throwin,['pass']


In [3]:
# Diccionario de la fuente primaria
dic_raw = pd.DataFrame([
    ("video_id",         "categórica (texto)", "ID del video = partido + mitad ('_0' / '_1')"),
    ("time",             "numérica continua (s)", "segundo del evento, absoluto en el video completo"),
    ("event",            "categórica nominal",  "play, throwin, challenge (eventos puntuales) | start, end (marcadores de jugada)"),
    ("event_attributes", "categórica (lista serializada)", "atributos del evento, p.ej. \"['pass', 'openplay']\"; vacío en start/end"),
], columns=["variable", "tipo", "descripción"])
dic_raw

,variable,tipo,descripción
0,video_id,categórica (texto),ID del video = partido + mitad ('_0' / '_1')
1,time,numérica continua (s),"segundo del evento, absoluto en el video completo"
2,event,categórica nominal,"play, throwin, challenge (eventos puntuales) |..."
3,event_attributes,categórica (lista serializada),"atributos del evento, p.ej. ""['pass', 'openpla..."


## 2. Las tablas que consume el equipo

### `events.csv` — una fila por evento anotado

In [4]:
print("Observaciones:", len(events))
print("Variables    :", events.shape[1])
print()
resumen_events = pd.DataFrame({
    "dtype": events.dtypes.astype(str),
    "n_no_nulos": events.notna().sum(),
    "n_nulos": events.isna().sum(),
    "%_nulos": (events.isna().mean() * 100).round(1),
    "n_unicos": events.nunique(),
})
resumen_events

Observaciones: 11218
Variables    : 11



,dtype,n_no_nulos,n_nulos,%_nulos,n_unicos
video_id,object,11218,0,0.0,12
time_s,float64,11218,0,0.0,11204
event,object,11218,0,0.0,5
event_attributes,object,4382,6836,60.9,14
attr_1,object,4382,6836,60.9,8
attr_2,object,3586,7632,68.0,3
is_point_event,bool,11218,0,0.0,2
clip_num,int64,11218,0,0.0,114
time_in_clip_s,float64,11218,0,0.0,7879
fps,float64,0,11218,100.0,0


In [5]:
# Tipo "conceptual" de cada variable de events.csv (para el informe)
dic_events = pd.DataFrame([
    ("video_id",       "categórica (12 niveles)", "ID del video"),
    ("time_s",         "numérica continua (s)",   "segundo del evento (= 'time' de la fuente)"),
    ("event",          "categórica nominal (5)",  "play / throwin / challenge / start / end"),
    ("event_attributes","texto (lista cruda)",    "se conserva como referencia"),
    ("attr_1",         "categórica nominal (8)",  "1er atributo: pass, cross, fouled, ..."),
    ("attr_2",         "categórica nominal (3)",  "2do atributo: openplay, freekick, corner"),
    ("is_point_event", "booleana",                "True si event in {play, throwin, challenge}"),
    ("clip_num",       "entero (índice)",         "clip de 30 s del mirror donde cae el evento"),
    ("time_in_clip_s", "numérica continua (s)",   "segundo del evento dentro de ese clip"),
    ("fps",            "numérica",                "fps del video; NaN si no se sondeó video"),
    ("frame_abs",      "entero",                  "floor(time_s * fps); NaN si no hay fps"),
], columns=["variable", "tipo", "descripción"])
dic_events

,variable,tipo,descripción
0,video_id,categórica (12 niveles),ID del video
1,time_s,numérica continua (s),segundo del evento (= 'time' de la fuente)
2,event,categórica nominal (5),play / throwin / challenge / start / end
3,event_attributes,texto (lista cruda),se conserva como referencia
4,attr_1,categórica nominal (8),"1er atributo: pass, cross, fouled, ..."
5,attr_2,categórica nominal (3),"2do atributo: openplay, freekick, corner"
6,is_point_event,booleana,"True si event in {play, throwin, challenge}"
7,clip_num,entero (índice),clip de 30 s del mirror donde cae el evento
8,time_in_clip_s,numérica continua (s),segundo del evento dentro de ese clip
9,fps,numérica,fps del video; NaN si no se sondeó video


In [6]:
core_raw = (raw.rename(columns={"time": "time_s"})[["video_id", "time_s", "event"]]
               .sort_values(["video_id", "time_s"]).reset_index(drop=True))
core_ev  = (events[["video_id", "time_s", "event"]]
               .sort_values(["video_id", "time_s"]).reset_index(drop=True))
print("filas raw   :", len(core_raw))
print("filas events:", len(core_ev))
print("misma cantidad de eventos por tipo:",
      raw["event"].value_counts().sort_index().equals(events["event"].value_counts().sort_index()))
print("max |Δ time_s| raw vs events:", (core_raw["time_s"] - core_ev["time_s"]).abs().max(),
      "  (round-off de float64 al escribir CSV)")

filas raw   : 11218
filas events: 11218
misma cantidad de eventos por tipo: True
max |Δ time_s| raw vs events: 4.547473508864641e-13   (round-off de float64 al escribir CSV)


### `plays.csv` — una fila por intervalo de jugada (`start` → `end`)

In [7]:
print("Observaciones:", len(plays))
print("Variables    :", plays.shape[1])
print()
resumen_plays = pd.DataFrame({
    "dtype": plays.dtypes.astype(str),
    "n_nulos": plays.isna().sum(),
    "n_unicos": plays.nunique(),
})
print(resumen_plays)
print()
plays.describe()

Observaciones: 3418
Variables    : 5

                  dtype  n_nulos  n_unicos
video_id         object        0        12
start_s         float64        0      3418
end_s           float64        0      3418
duration_s      float64        0       641
n_point_events    int64        0         6



,start_s,end_s,duration_s,n_point_events
count,3418.000000,3418.000000,3418.000000,3418.000000
mean,1792.441793,1795.494906,3.053118,1.282036
std,862.394643,862.346547,1.164978,0.573566
min,175.025822,177.525822,2.500000,1.000000
25%,1051.411681,1054.800270,2.500000,1.000000
50%,1775.006050,1779.029948,2.500000,1.000000
75%,2533.470481,2535.970481,2.500000,1.000000
max,3572.500727,3575.000727,13.990000,6.000000


### `video_meta.csv` — una fila por video

In [8]:
print("Observaciones:", len(video_meta), " Variables:", video_meta.shape[1])
print(video_meta.dtypes)
video_meta

Observaciones: 12  Variables: 8
video_id             object
has_video              bool
n_clips               int64
n_events              int64
n_play                int64
n_throwin             int64
n_challenge           int64
annot_time_max_s    float64
dtype: object


,video_id,has_video,n_clips,n_events,n_play,n_throwin,n_challenge,annot_time_max_s
0,1606b0e6_0,True,76,1000,319,21,56,3014.7
1,1606b0e6_1,True,92,1249,434,13,60,3387.7
2,35bd9041_0,True,83,1075,355,8,48,3061.9
3,35bd9041_1,True,93,933,290,12,57,3477.7
4,3c993bd2_0,True,81,1042,347,14,53,2999.0
5,3c993bd2_1,False,0,966,289,17,70,3472.4
6,407c5a9e_1,False,0,858,283,11,56,3445.2
7,4ffd5986_0,False,0,792,235,10,57,3099.6
8,9a97dae4_1,False,0,737,237,6,48,2999.7
9,cfbe2e94_0,False,0,823,241,29,35,3216.9


## 3. Conteos base y tabla de frecuencias

12 videos (8 partidos: 4 con las dos mitades, 4 con una sola), **11 218 eventos** en total.

In [9]:
print("Videos anotados :", events["video_id"].nunique())
print("Partidos        :", len({v[:-2] for v in events['video_id']}))
print("Eventos totales :", len(events))
print()
print("Frecuencia de 'event':")
freq = events["event"].value_counts()
tabla = pd.DataFrame({"n": freq, "proporción": (freq / len(events)).round(4)})
print(tabla)
print()
pe = events[events["is_point_event"]]
print(f"Eventos puntuales: {len(pe)}  ({len(pe)/len(events):.1%})")
print("  desbalance play : throwin =", round((events.event=='play').sum() / (events.event=='throwin').sum(), 1), ": 1")

Videos anotados : 12
Partidos        : 8
Eventos totales : 11218

Frecuencia de 'event':
              n  proporción
event                      
play       3586      0.3197
start      3418      0.3047
end        3418      0.3047
challenge   624      0.0556
throwin     172      0.0153

Eventos puntuales: 4382  (39.1%)
  desbalance play : throwin = 20.8 : 1


In [10]:
print("attr_1 por tipo de evento puntual:")
for e in ["play", "throwin", "challenge"]:
    print(f"\n  {e}:")
    print(events.loc[events.event == e, "attr_1"].value_counts().to_string())

print("\n\nattr_2 (solo play / throwin):")
print(events["attr_2"].value_counts(dropna=False).to_string())

attr_1 por tipo de evento puntual:

  play:
attr_1
pass     3468
cross     118

  throwin:
attr_1
pass     154
cross     18

  challenge:
attr_1
ball_action_forced                239
opponent_dispossessed             138
fouled                            111
challenge_during_ball_transfer     53
possession_retained                44
opponent_rounded                   39


attr_2 (solo play / throwin):
attr_2
NaN         7632
openplay    3417
freekick     132
corner        37


In [11]:
# Eventos por video
por_video = video_meta.set_index("video_id")[["n_events", "n_play", "n_throwin", "n_challenge", "has_video"]]
print(por_video)
print("\nn_events por video -- describe:")
print(video_meta["n_events"].describe().round(1))

            n_events  n_play  n_throwin  n_challenge  has_video
video_id                                                       
1606b0e6_0      1000     319         21           56       True
1606b0e6_1      1249     434         13           60       True
35bd9041_0      1075     355          8           48       True
35bd9041_1       933     290         12           57       True
3c993bd2_0      1042     347         14           53       True
3c993bd2_1       966     289         17           70      False
407c5a9e_1       858     283         11           56      False
4ffd5986_0       792     235         10           57      False
9a97dae4_1       737     237          6           48      False
cfbe2e94_0       823     241         29           35      False
cfbe2e94_1       763     232         15           38      False
ecf251d4_0       980     324         16           46      False

n_events por video -- describe:
count      12.0
mean      934.8
std       149.1
min       737.0
25%    

## 4. Valores faltantes

Hay que separar dos casos:

- **Faltante estructural (esperado):** `event_attributes`, `attr_1`, `attr_2`, `fps`, `frame_abs`.
  Los marcadores `start` / `end` **no tienen** atributos por definición (6 836 filas = 3 418 + 3 418),
  y `fps` / `frame_abs` quedan en `NaN` porque la versión versionada de `events.csv` se generó
  **sin sondear los videos** (no se necesita para el EDA de anotaciones).
- **Faltante real (problema de datos):** ninguno. Los **4 382 eventos puntuales tienen
  atributo completo** (0 faltantes), `time_s` no tiene nulos ni negativos, y no hay filas
  duplicadas.

In [12]:
print("Nulos por columna en events.csv:")
print(events.isna().sum())
print()

nulos_attr1 = events["attr_1"].isna()
print("filas con attr_1 nulo :", nulos_attr1.sum())
print("de esas, start/end    :", events.loc[nulos_attr1, "event"].isin(["start", "end"]).sum())
print("-> el faltante de atributos es 100% estructural (start/end)")
print()

print("Faltante REAL en eventos puntuales:")
falta_attr = pe["event_attributes"].isna() | pe["event_attributes"].astype(str).str.strip().isin(["", "[]"])
print(pe.assign(falta=falta_attr).groupby("event")["falta"].sum())
print()
print("time_s negativos :", (events["time_s"] < 0).sum())
print("time_s nulos     :", events["time_s"].isna().sum())
print("filas duplicadas :", events.duplicated().sum())

Nulos por columna en events.csv:
video_id                0
time_s                  0
event                   0
event_attributes     6836
attr_1               6836
attr_2               7632
is_point_event          0
clip_num                0
time_in_clip_s          0
fps                 11218
frame_abs           11218
dtype: int64

filas con attr_1 nulo : 6836
de esas, start/end    : 6836
-> el faltante de atributos es 100% estructural (start/end)

Faltante REAL en eventos puntuales:
event
challenge    0
play         0
throwin      0
Name: falta, dtype: int64

time_s negativos : 0
time_s nulos     : 0
filas duplicadas : 0


In [13]:
print("fps nulos      :", events['fps'].isna().sum(), f"({events['fps'].isna().mean():.0%})")
print("frame_abs nulos:", events['frame_abs'].isna().sum(), f"({events['frame_abs'].isna().mean():.0%})")
print()
print("Decisión: se dejan en NaN.")


fps nulos      : 11218 (100%)
frame_abs nulos: 11218 (100%)

Decisión: se dejan en NaN.


## Emparejamiento de marcadores `start` / `end`

Cada jugada se anota con un `start` y un `end`. 

In [14]:
se = events[events["event"].isin(["start", "end"])]
print("start:", (se.event == "start").sum())
print("end  :", (se.event == "end").sum())

unmatched = []
for vid, g in se.groupby("video_id"):
    open_start = None
    for _, r in g.sort_values("time_s").iterrows():
        if r["event"] == "start":
            if open_start is not None:
                unmatched.append((vid, open_start, "start_sin_end"))
            open_start = r["time_s"]
        else:
            if open_start is None:
                unmatched.append((vid, r["time_s"], "end_sin_start"))
            else:
                open_start = None
    if open_start is not None:
        unmatched.append((vid, open_start, "start_sin_end"))

print("\nMarcadores sin pareja:", len(unmatched))
print("Archivo plays_unmatched.csv presente:", (PROC / "plays_unmatched.csv").exists())
print("Jugadas emparejadas en plays.csv    :", len(plays), "(= nº de 'start' = nº de 'end')")
print("\n-> Emparejamiento perfecto: 3 418 start, 3 418 end, 0 sueltos, 0 solapamientos.")

start: 3418
end  : 3418

Marcadores sin pareja: 0
Archivo plays_unmatched.csv presente: False
Jugadas emparejadas en plays.csv    : 3418 (= nº de 'start' = nº de 'end')

-> Emparejamiento perfecto: 3 418 start, 3 418 end, 0 sueltos, 0 solapamientos.


In [15]:
# Verificar que no hay solapamiento entre jugadas de un mismo video
solapan = 0
for vid, g in plays.sort_values("start_s").groupby("video_id"):
    prev_end = -np.inf
    for _, r in g.iterrows():
        if r["start_s"] < prev_end:
            solapan += 1
        prev_end = r["end_s"]
print("jugadas que solapan con la anterior:", solapan)

print("\nn_point_events por jugada (toda jugada contiene >= 1 evento puntual):")
print(plays["n_point_events"].value_counts().sort_index())
print("\nduration_s -- una jugada 'canónica' dura 2.5 s (start = evento - 2 s aprox, end = evento + 0.5 s):")
print("  == 2.5 s :", (plays["duration_s"] == 2.5).sum())
print("  >  2.5 s :", (plays["duration_s"] > 2.5).sum(), "(jugadas encadenadas: varios eventos seguidos)")
print("  máx      :", plays["duration_s"].max(), "s")

jugadas que solapan con la anterior: 0

n_point_events por jugada (toda jugada contiene >= 1 evento puntual):
n_point_events
1    2622
2     666
3     101
4      21
5       7
6       1
Name: count, dtype: int64

duration_s -- una jugada 'canónica' dura 2.5 s (start = evento - 2 s aprox, end = evento + 0.5 s):
  == 2.5 s : 2622
  >  2.5 s : 796 (jugadas encadenadas: varios eventos seguidos)
  máx      : 13.99 s


## 6. Cobertura de video: solo 5 de 12, y parcial

`has_video` es `True` para **5 videos**; para los otros 7 el mirror no trae nada
(`n_clips = 0`). Además, para los 5 que sí tienen footage, los clips no llegan hasta el
final del partido: las anotaciones se extienden más allá del último clip disponible.

In [16]:
print("Videos con footage :", int(video_meta["has_video"].sum()), "/", len(video_meta))
print("Clips totales      :", len(clip_index), "en", clip_index["video_id"].nunique(), "videos")
print()

cob = video_meta.loc[video_meta["has_video"], ["video_id", "n_clips", "annot_time_max_s"]].copy()
cob["video_cubre_s"] = cob["n_clips"] * 30.0
cob["gap_s"] = (cob["annot_time_max_s"] - cob["video_cubre_s"]).round(1)

ev_pt = events[events["is_point_event"]]
cob["ev_puntuales"] = cob["video_id"].map(ev_pt.groupby("video_id").size())
cob["ev_fuera_de_video"] = cob["video_id"].map(
    ev_pt.assign(fuera=ev_pt["clip_num"] > ev_pt["video_id"].map(video_meta.set_index("video_id")["n_clips"]))
         .groupby("video_id")["fuera"].sum())
cob["%_ev_sin_video"] = (cob["ev_fuera_de_video"] / cob["ev_puntuales"] * 100).round(1)
cob.reset_index(drop=True)

Videos con footage : 5 / 12
Clips totales      : 425 en 5 videos



,video_id,n_clips,annot_time_max_s,video_cubre_s,gap_s,ev_puntuales,ev_fuera_de_video,%_ev_sin_video
0,1606b0e6_0,76,3014.7,2280.0,734.7,396,103,26.0
1,1606b0e6_1,92,3387.7,2760.0,627.7,507,126,24.9
2,35bd9041_0,83,3061.9,2490.0,571.9,411,99,24.1
3,35bd9041_1,93,3477.7,2790.0,687.7,359,66,18.4
4,3c993bd2_0,81,2999.0,2430.0,569.0,414,79,19.1


In [17]:
tot_pe = len(ev_pt)
pe_con_video = ev_pt["video_id"].isin(video_meta.loc[video_meta.has_video, "video_id"])
pe_dentro = pe_con_video & (ev_pt["clip_num"] <= ev_pt["video_id"].map(video_meta.set_index("video_id")["n_clips"]))
print(f"Eventos puntuales totales                    : {tot_pe}")
print(f"  en un video del que hay footage            : {pe_con_video.sum()}  ({pe_con_video.mean():.1%})")
print(f"  y ademas dentro de un clip disponible      : {pe_dentro.sum()}  ({pe_dentro.mean():.1%})")
print()


Eventos puntuales totales                    : 4382
  en un video del que hay footage            : 2087  (47.6%)
  y ademas dentro de un clip disponible      : 1614  (36.8%)



## 7. Operaciones de limpieza / preprocesamiento aplicadas


| # | Operación | Detalle | Filas afectadas |
|---|---|---|---|
| 1 | **Renombrar** `time` → `time_s` | claridad de unidades | todas |
| 2 | **Explotar** `event_attributes` | `"['pass','openplay']"` → `attr_1='pass'`, `attr_2='openplay'` (máx. 2 atributos) | 4 382 eventos puntuales |
| 3 | **Derivar** `is_point_event` | `event ∈ {play, throwin, challenge}` | todas |
| 4 | **Mapear** evento → clip | `clip_num = ⌊time_s / 30⌋ + 1`, `time_in_clip_s = time_s mod 30` | todas |
| 5 | **Ordenar** por `(video_id, time_s)` | | todas |
| 6 | **Emparejar** `start`/`end` → `plays.csv` | intervalos + `duration_s` + `n_point_events`; sueltos → `plays_unmatched.csv` | 3 418 jugadas |
| 7 | **Sondear** video (opcional, con `cv2`) | `fps`, `width`, `height`; `frame_abs = ⌊time_s·fps⌋` | solo si hay footage |
| 8 | **Construir** `video_meta.csv` / `clip_index.csv` | conteos por video, índice de clips | 12 / 5 videos |

No se eliminaron filas, no se imputaron valores y no se detectaron duplicados ni registros
corruptos: la fuente primaria ya venía consistente. La "limpieza" es esencialmente
**normalización de formato** (una lista de strings → columnas tipadas) y **derivación de
tablas** para el análisis.

In [18]:
# Chequeos de integridad finales (todo debe dar True / 0)
checks = {
    "events: sin nulos en time_s":            events["time_s"].notna().all(),
    "events: sin time_s negativos":            (events["time_s"] >= 0).all(),
    "events: sin duplicados":                  events.duplicated().sum() == 0,
    "events: eventos puntuales con attr_1":    events.loc[events.is_point_event, "attr_1"].notna().all(),
    "events: nº filas == fuente primaria":     len(events) == len(raw),
    "plays: nº jugadas == nº de 'start'":      len(plays) == (events.event == "start").sum(),
    "plays: duration_s > 0":                   (plays["duration_s"] > 0).all(),
    "plays: toda jugada con >=1 evento":       (plays["n_point_events"] >= 1).all(),
    "video_meta: 12 videos":                   len(video_meta) == 12,
    "video_meta: 5 con footage":               int(video_meta["has_video"].sum()) == 5,
    "video_meta: Σ n_events == len(events)":   video_meta["n_events"].sum() == len(events),
}
for k, v in checks.items():
    print(f"  [{'OK ' if v else 'XX '}] {k}: {v}")
assert all(checks.values()), "Falló un chequeo de integridad"
print("\nTodos los chequeos de integridad pasan.")

  [OK ] events: sin nulos en time_s: True
  [OK ] events: sin time_s negativos: True
  [OK ] events: sin duplicados: True
  [OK ] events: eventos puntuales con attr_1: True
  [OK ] events: nº filas == fuente primaria: True
  [OK ] plays: nº jugadas == nº de 'start': True
  [OK ] plays: duration_s > 0: True
  [OK ] plays: toda jugada con >=1 evento: True
  [OK ] video_meta: 12 videos: True
  [OK ] video_meta: 5 con footage: True
  [OK ] video_meta: Σ n_events == len(events): True

Todos los chequeos de integridad pasan.


## 8. Hallazgos para la Descripción de los datos

1. **Fuente única y derivación limpia.** Todo sale de `train_annotations.csv` (11 218 filas,
   4 columnas). Las tablas `events` / `plays` / `video_meta` son derivadas y trazables; la
   normalización es 1:1 (sin borrar ni imputar filas).
2. **Estructura.** `events.csv`: 11 218 obs. × 11 vars (1 continua de tiempo, 1 booleana,
   3 categóricas de contenido, resto índices/derivadas). `plays.csv`: 3 418 obs. × 5 vars.
   `video_meta.csv`: 12 obs. × 8 vars.
3. **Sin faltantes reales.** Los únicos `NaN` son estructurales: atributos en `start`/`end`
   y `fps`/`frame_abs` (no se sondeó video en la versión versionada). Los 4 382 eventos
   puntuales tienen atributo completo.
4. **Marcadores de jugada perfectamente emparejados:** 3 418 `start` y 3 418 `end`, cero
   sueltos, cero solapamientos. Cada jugada contiene 1 o mas eventos puntuales; la duración
   canónica es 2.5 s (796 jugadas duran más por eventos encadenados).
5. **Desbalance de clases fuerte** (relevante para el modelado): `play` ≈ 21× `throwin`,
   y `challenge` ≈ 3.6× `throwin`. `throwin` es la clase minoritaria con solo 172 casos.
6. **Cobertura de video pobre:** solo **5 de 12** videos tienen footage, y para esos 5 el
   mirror no cubre el partido completo — apenas ~1/3 de los eventos puntuales caen dentro
   de un clip disponible. El análisis visual (Persona 4) queda limitado a ese subconjunto;
   el análisis de anotaciones (Personas 2 y 3) usa los 12 videos sin problema.